In [ ]:
!sudo apt update && sudo apt install libassimp-dev
!pip install torch
!pip install --upgrade hyperdrone[examples]

In [ ]:
import math
import time

import numpy as np
from scipy.spatial.transform import Rotation

from hyperdrone import render
from hyperdrone.examples.data import procthor_scene_path



## Benchmark

In [ ]:
NUM_CAMERAS = 4096
WIDTH, HEIGHT = 64, 64
FOV = 1.3962634015954636  # 80 degrees, the renderer config default
EYE = np.array([-3.92, -5.67, 1.0])
WARMUP_SECONDS, SECONDS, SYNC_INTERVAL = 2.0, 10.0, 10

scene = render.load_scene(procthor_scene_path(), fidelity="medium") # load any GLB
renderer = render.Renderer(width=WIDTH, height=HEIGHT, num_cameras=NUM_CAMERAS, num_probes=1,
                           output="rgb", fidelity="medium")
renderer.init(scene)

def normalize(vectors):
    return vectors / np.linalg.norm(vectors, axis=-1, keepdims=True)


rotations = Rotation.random(NUM_CAMERAS, rng=np.random.default_rng(0)).as_matrix()
forwards = rotations[:, :, 0]  # camera forward = body +X column
ups = rotations[:, :, 2]       # camera up = body +Z column

scale = 2.0 * math.tan(FOV / 2.0)
du = normalize(np.cross(forwards, ups)) * scale
dv = normalize(np.cross(du, forwards)) * scale
dir_00 = forwards - 0.5 * du + 0.5 * dv
positions = np.tile(EYE, (NUM_CAMERAS, 1))
renderer.set_cameras(np.concatenate([positions, dir_00, du, -dv], axis=-1).astype(np.float32))
renderer.generate_probe_directions()

warmup_start = time.perf_counter()
while time.perf_counter() - warmup_start < WARMUP_SECONDS:
    renderer.render()
renderer.synchronize()

wall_start = time.perf_counter()
num_iterations = 0
while True:
    renderer.render_launch()
    num_iterations += 1
    if num_iterations % SYNC_INTERVAL == 0:
        renderer.render_sync()
        if time.perf_counter() - wall_start >= SECONDS:
            break
renderer.synchronize()
wall_seconds = time.perf_counter() - wall_start

total_frames = num_iterations * NUM_CAMERAS
print(f"Backend:         {renderer.backend}")
print(f"Iterations:      {num_iterations}")
print(f"Wall-clock time: {wall_seconds * 1000:.2f} ms")
print(f"Avg per iter:    {wall_seconds * 1000 / num_iterations:.2f} ms")
print(f"Throughput:      {total_frames / wall_seconds:.1f} frames/sec")
print(f"Total MRays/sec: {total_frames * WIDTH * HEIGHT / 1e6 / wall_seconds:.1f}")

## Display Outputs

In [ ]:
import torch, matplotlib.pyplot as plt
frame = torch.from_dlpack(renderer.frame_dlpack()) # zero-copy access
print(f'Framebuffer device: "{frame.device}"')
print(f'Framebuffer shape: "{frame.shape}"')

DISPLAY_GRID = 12
grid_frames = frame[:DISPLAY_GRID**2].view(DISPLAY_GRID, DISPLAY_GRID, HEIGHT, WIDTH, 4)
tiled_image = grid_frames.permute(0, 2, 1, 3, 4).reshape(DISPLAY_GRID * HEIGHT, DISPLAY_GRID * WIDTH, 4)

plt.figure(figsize=(12, 12))
plt.imshow(tiled_image.cpu())
plt.axis('off')
plt.show()